#### July 29, 2025, Liam Tran
### Multiple Correspondence Analysis Testing

# Functions

In [1]:
from sklearn.decomposition import PCA
import prince
import mca

import sys, common, pickle, collections, math, numpy, scipy.stats
from pyx import *
from icecream import ic
import pandas as pd

def parsePickleFile(pickleFile, barcodeDict=None):
    """
    dataDict is of format:
    {readID:{edit_string,barcode,query_string,ref_string,aligned_pairs:...}}
    Will sort reads according to barcode, and output a tally of the number
    of reads per barcode. Will also compute the total size (in nts) of each
    barcode, to get a sense of coverage.
    
    Will output a dictionary of format:
    {bc:{readID:{position:edit}}}
    """
    ##start by unpickling the input file
    with open(pickleFile,'rb') as f:
        dataDict=pickle.load(f)
    ##
    aa=collections.defaultdict(lambda:collections.defaultdict(int))
    for readID,subDict in dataDict.items():
        bc=subDict['barcode']
        if barcodeDict is not None:
            if bc not in barcodeDict:
                continue
        queryLength=len(subDict['read_sequence'])
        aa[bc]['ct']+=1
        aa[bc]['length']+=queryLength
    ##
    for k,v in aa.items():
        print('Barcode %s had %s sequences totaling %s kilo nts.'%(\
            k,v['ct'],v['length']/1000))
    ##
    bb=collections.defaultdict(lambda:collections.defaultdict(dict))
    ##
    for readID,subDict in dataDict.items():
        barCode=subDict['barcode']
        #print(barCode,readID)
        editString=subDict['edit_string']
        #print(editString,len(editString))
        readSeq=subDict['read_sequence_aligned']
        #print(readSeq,len(readSeq))
        refSeq=subDict['ref_sequence_aligned']
        #print(refSeq,len(refSeq))
        alignedPairs=subDict['aligned_pairs']
        #print(alignedPairs,len(alignedPairs))
        #sys.exit()
        for ii in range(len(alignedPairs)-1):
            entry=alignedPairs[ii]
            idx=entry[0]
            absIdx=entry[1]
            if idx!=None and absIdx!=None:
                seq=refSeq[ii]
                edit=editString[ii]
                if seq=='A' and edit!='2' and absIdx<=704:
                    ##695 is where the RT primer binds--anything past this
                    ##is artifact.
                    bb[barCode][readID][absIdx]=int(edit)
    ##
    return bb

def parseBarcodeFile(barcodeFile):
    '''
    Given a barcode txt file, will return a dict of
    {barcode:library_name}
    '''
    bcDict={}
    with open(barcodeFile,'r') as f:
        for line in f:
            line=line.rstrip()
            print(line)
            if line=='':
                continue
            parts=line.split(',')
            bc=parts[0]
            libName=parts[1]
            bcDict[bc]=libName
    ##
    return bcDict

def make_data_frame(read_dict, positions):
    """
    Convert the read dictionary into a DataFrame suitable for MCA.
    Each row corresponds to a read, and each column corresponds to a position in the reference sequence. 

    I will also add a 'barcode' column to the DataFrame for grouping purposes.
    """
    import pandas as pd
    data = []
    

    for bc, reads in read_dict.items():
        for readID, positions in reads.items():
            # Create a binary string for each read
            binary_string = [0] * 696  # Assuming positions are from 0 to 695
            for pos, editStatus in positions.items():
                binary_string[pos] = editStatus
            # Append the barcode and binary string to the data list
            data.append([readID]+ [bc] + binary_string)
            
    # drop positions that are not adenosine
    

    # Create a DataFrame
    columns = ['readID', 'barcode'] + [f'pos_{i}' for i in range(696)]
    df = pd.DataFrame(data, columns=columns)
    
    return df

def get_adenosine_positions(ref_seq):
    """
    Get the positions of adenosine in the reference sequence.
    Returns a list of positions where adenosine is present.
    """
    adenosine_positions = []
    for i, base in enumerate(ref_seq.upper()):
        if base == 'A':
            adenosine_positions.append(i)
    return adenosine_positions

# Load Data

In [ ]:
from pathlib import Path
from Bio import SeqIO

# read_dict = parsePickleFile('/data16/liam/working/data/alignments/250509_JANP-112_LT/pickles/merged_barcodes_nanoluc_translatable.pickle')
read_dict = parsePickleFile('/data16/liam/working/data/alignments/251030_JANP-132_LT/pickles/five_percent_full.sorted_nanoluc.pickle')

Barcode bc1 had 8051 sequences totaling 5881.605 kilo nts.
Barcode bc2 had 13977 sequences totaling 10231.988 kilo nts.
Barcode bc3 had 31286 sequences totaling 22850.76 kilo nts.
Barcode bc4 had 54113 sequences totaling 39409.364 kilo nts.
Barcode bc5 had 92157 sequences totaling 67328.657 kilo nts.


In [6]:
reference = Path('/data16/liam/genomes/251005_3xFLAG_nLuc.fa')
ref_dict = SeqIO.to_dict(SeqIO.parse(reference, 'fasta'))

adeninePositions = get_adenosine_positions(ref_dict['nanoluc'].seq)

IndexError: list assignment index out of range